In [5]:
# ============================================================
# INSURANCE FRAUD FEATURE ENGINEERING PROJECT
# FINAL FULL CORRECTED VERSION
# ============================================================

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# =========================
# LOAD DATASETS
# =========================

claims = pd.read_csv('claims_cleaned.csv')

policies = pd.read_csv('policies_cleaned.csv')

providers = pd.read_csv('providers_cleaned.csv')

agents = pd.read_csv('agents_cleaned.csv')

complaints = pd.read_csv('customer_complaints_cleaned.csv')

investigations = pd.read_csv('claim_investigations_cleaned.csv')

payments = pd.read_csv('payments_cleaned.csv')

renewals = pd.read_csv('policy_renewals_cleaned.csv')

branches = pd.read_csv('branches_cleaned.csv')



# ============================================================
# CLEAN KEY COLUMNS
# ============================================================

claims['policy_id'] = (
    claims['policy_id']
    .astype(str)
    .str.strip()
)

policies['policy_id'] = (
    policies['policy_id']
    .astype(str)
    .str.strip()
)

claims['provider_id'] = (
    claims['provider_id']
    .astype(str)
    .str.strip()
)

providers['provider_id'] = (
    providers['provider_id']
    .astype(str)
    .str.strip()
)

policies['agent_id'] = (
    policies['agent_id']
    .astype(str)
    .str.strip()
)

agents['agent_id'] = (
    agents['agent_id']
    .astype(str)
    .str.strip()
)

# ============================================================
# HANDLE MISSING VALUES
# ============================================================

def fill_missing_values(df):

    for col in df.columns:

        # Numeric Columns
        if pd.api.types.is_numeric_dtype(df[col]):

            df[col] = df[col].fillna(0)

        # Categorical Columns
        else:

            df[col] = df[col].fillna('Unknown')

    return df


claims = fill_missing_values(claims)

policies = fill_missing_values(policies)

providers = fill_missing_values(providers)

agents = fill_missing_values(agents)

complaints = fill_missing_values(complaints)

investigations = fill_missing_values(investigations)

payments = fill_missing_values(payments)

renewals = fill_missing_values(renewals)

branches = fill_missing_values(branches)

# ============================================================
# FIX DATATYPES BEFORE MERGE
# ============================================================

claims['policy_id'] = claims['policy_id'].astype(str)

policies['policy_id'] = policies['policy_id'].astype(str)

claims['provider_id'] = claims['provider_id'].astype(str)

providers['provider_id'] = providers['provider_id'].astype(str)

policies['agent_id'] = policies['agent_id'].astype(str)

agents['agent_id'] = agents['agent_id'].astype(str)

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

claims['incident_date'] = pd.to_datetime(
    claims['incident_date'],
    errors='coerce'
)

claims['claim_submission_date'] = pd.to_datetime(
    claims['claim_submission_date'],
    errors='coerce'
)

policies['policy_start_date'] = pd.to_datetime(
    policies['policy_start_date'],
    errors='coerce'
)

policies['policy_end_date'] = pd.to_datetime(
    policies['policy_end_date'],
    errors='coerce'
)

complaints['complaint_date'] = pd.to_datetime(
    complaints['complaint_date'],
    errors='coerce'
)

# ============================================================
# REMOVE DUPLICATE COLUMNS BEFORE MERGE
# ============================================================

duplicate_cols = [
    'coverage_amount',
    'policy_start_date',
    'agent_id'
]

for col in duplicate_cols:

    if col in claims.columns:

        claims.drop(columns=col, inplace=True)

# ============================================================
# MERGE POLICY TABLE
# ============================================================

claims = claims.merge(

    policies[
        [
            'policy_id',
            'coverage_amount',
            'policy_start_date',
            'agent_id'
        ]
    ],

    on='policy_id',
    how='left'

)

# ============================================================
# FILL MISSING VALUES AFTER MERGE
# ============================================================

claims['coverage_amount'] = claims['coverage_amount'].fillna(

    claims['coverage_amount'].median()

)

claims['agent_id'] = claims['agent_id'].fillna('Unknown')

# ============================================================
# FEATURE ENGINEERING
# ============================================================

# ============================================================
# 1. claim_month
# ============================================================

claims['claim_month'] = (

    claims['claim_submission_date']

    .dt.month

)

# ============================================================
# 2. claim_day_name
# ============================================================

claims['claim_day_name'] = (

    claims['claim_submission_date']

    .dt.day_name()

)

# ============================================================
# 3. policy_age_days_at_claim
# ============================================================

claims['policy_age_days_at_claim'] = (

    claims['claim_submission_date']

    -

    claims['policy_start_date']

).dt.days

# Replace negative values with NaN

claims['policy_age_days_at_claim'] = np.where(

    claims['policy_age_days_at_claim'] < 0,

    np.nan,

    claims['policy_age_days_at_claim']

)

# Fill NaN with median

claims['policy_age_days_at_claim'] = (

    claims['policy_age_days_at_claim']

    .fillna(

        claims['policy_age_days_at_claim'].median()

    )

)

# ============================================================
# 4. claim_to_coverage_ratio
# ============================================================

claims['claim_to_coverage_ratio'] = np.where(

    claims['coverage_amount'] > 0,

    claims['claim_amount']

    /

    claims['coverage_amount'],

    0

)

# ============================================================
# 5. high_value_claim_flag
# ============================================================

claims['high_value_claim_flag'] = np.where(

    claims['claim_amount'] > 2000000,

    1,

    0

)

# ============================================================
# 6. early_claim_flag
# ============================================================

claims['early_claim_flag'] = np.where(

    claims['policy_age_days_at_claim'] <= 30,

    1,

    0

)

# ============================================================
# 7. duplicate_claim_flag
# ============================================================

claims['duplicate_claim_flag'] = claims.duplicated(

    subset=[

        'customer_id',

        'claim_amount',

        'claim_submission_date'

    ],

    keep=False

).astype(int)

# ============================================================
# 8. customer_claim_count
# ============================================================

claims['customer_claim_count'] = (

    claims.groupby('customer_id')['claim_id']

    .transform('count')

)

# ============================================================
# 9. frequent_claim_customer_flag
# ============================================================

claims['frequent_claim_customer_flag'] = np.where(

    claims['customer_claim_count'] >= 3,

    1,

    0

)

# ============================================================
# 10. provider_high_risk_flag
# ============================================================

provider_claim_counts = (

    claims.groupby('provider_id')['claim_id']

    .count()

)

high_risk_providers = provider_claim_counts[

    provider_claim_counts >=

    provider_claim_counts.quantile(0.90)

].index

claims['provider_high_risk_flag'] = np.where(

    claims['provider_id'].isin(high_risk_providers),

    1,

    0

)

# ============================================================
# 11. agent_high_risk_flag
# ============================================================

agent_claim_counts = (

    claims.groupby('agent_id')['claim_id']

    .count()

)

high_risk_agents = agent_claim_counts[

    agent_claim_counts >=

    agent_claim_counts.quantile(0.90)

].index

claims['agent_high_risk_flag'] = np.where(

    claims['agent_id'].isin(high_risk_agents),

    1,

    0

)

# ============================================================
# 12. settlement_days
# ============================================================

# Using already available settlement_days column
# because claims table already contains it

claims['settlement_days'] = pd.to_numeric(

    claims['settlement_days'],

    errors='coerce'

)

# Replace negative values with NaN

claims['settlement_days'] = np.where(

    claims['settlement_days'] < 0,

    np.nan,

    claims['settlement_days']

)

# Fill NaN with median

claims['settlement_days'] = (

    claims['settlement_days']

    .fillna(

        claims['settlement_days'].median()

    )

)

# ============================================================
# 13. settlement_delay_flag
# ============================================================

claims['settlement_delay_flag'] = np.where(

    claims['settlement_days'] > 45,

    1,

    0

)

# ============================================================
# 14. complaint_after_claim_flag
# ============================================================

latest_complaint = (

    complaints.groupby(

        'customer_id',

        as_index=False

    )['complaint_date']

    .max()

)

claims = claims.merge(

    latest_complaint,

    on='customer_id',

    how='left'

)

claims['complaint_after_claim_flag'] = np.where(

    claims['complaint_date']

    >

    claims['claim_submission_date'],

    1,

    0

)

# ============================================================
# 15. provider_risk_score
# ============================================================

claims['provider_risk_score'] = (

    claims.groupby('provider_id')['fraud_flag']

    .transform('mean')

    * 100

)

claims['provider_risk_score'] = (

    claims['provider_risk_score']

    .round(2)

)

# ============================================================
# 16. customer_risk_score
# ============================================================

claims['customer_risk_score'] = (

    (claims['frequent_claim_customer_flag'] * 25)

    +

    (claims['duplicate_claim_flag'] * 25)

    +

    (claims['high_value_claim_flag'] * 20)

    +

    (claims['early_claim_flag'] * 15)

    +

    (claims['complaint_after_claim_flag'] * 15)

)

# ============================================================
# 17. claim_risk_score
# ============================================================

claims['claim_risk_score'] = (

    (claims['early_claim_flag'] * 15)

    +

    (claims['high_value_claim_flag'] * 20)

    +

    (claims['duplicate_claim_flag'] * 20)

    +

    (claims['provider_high_risk_flag'] * 15)

    +

    (claims['agent_high_risk_flag'] * 10)

    +

    (claims['frequent_claim_customer_flag'] * 10)

    +

    (claims['settlement_delay_flag'] * 10)

)

# ============================================================
# 18. risk_category
# ============================================================

claims['risk_category'] = np.where(

    claims['claim_risk_score'] >= 70,

    'High Risk',

    np.where(

        claims['claim_risk_score'] >= 40,

        'Medium Risk',

        'Low Risk'

    )

)

# ============================================================
# FINAL MISSING VALUE CHECK
# ============================================================

for col in claims.columns:

    if pd.api.types.is_numeric_dtype(claims[col]):

        claims[col] = claims[col].fillna(0)

    else:

        claims[col] = claims[col].fillna('Unknown')

# ============================================================
# VALIDATION
# ============================================================

print('\nDataset Shape:')
print(claims.shape)

print('\nMissing Values:')
print(claims.isnull().sum())

print('\nDuplicate Claim IDs:')
print(claims['claim_id'].duplicated().sum())

print('\nNegative Settlement Days:')
print((claims['settlement_days'] < 0).sum())

print('\nNegative Policy Age:')
print((claims['policy_age_days_at_claim'] < 0).sum())

print('\nFeature Engineering Completed Successfully')

# ============================================================
# SAVE FINAL DATASET
# ============================================================

claims.to_csv(

    'insurance_claims_feature_engineered_v4.csv',

    index=False

)

print('\nFinal Dataset Saved Successfully')

C:\Users\RAVITEJA.P\AppData\Local\Temp\ipykernel_18872\3659894115.py:146: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  policies['policy_start_date'] = pd.to_datetime(
C:\Users\RAVITEJA.P\AppData\Local\Temp\ipykernel_18872\3659894115.py:151: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  policies['policy_end_date'] = pd.to_datetime(



Dataset Shape:
(151014, 32)

Missing Values:
claim_id                        0
policy_id                       0
customer_id                     0
claim_type                      0
claim_amount                    0
claim_submission_date           0
incident_date                   0
claim_status                    0
fraud_flag                      0
settlement_days                 0
provider_id                     0
coverage_amount                 0
policy_start_date               0
agent_id                        0
claim_month                     0
claim_day_name                  0
policy_age_days_at_claim        0
claim_to_coverage_ratio         0
high_value_claim_flag           0
early_claim_flag                0
duplicate_claim_flag            0
customer_claim_count            0
frequent_claim_customer_flag    0
provider_high_risk_flag         0
agent_high_risk_flag            0
settlement_delay_flag           0
complaint_date                  0
complaint_after_claim_flag      0
pr